# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as object attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets and their fields
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record Set: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        column_ids = []
        if hasattr(field, 'columns') and field.columns:
            column_ids = [col.id for col in field.columns if hasattr(col, 'id')]
        print(f"    - Field: {field.name} (@id: {field.id}) | Columns: {column_ids}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, extract all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        dataframes[rs_id] = pd.DataFrame()  # Empty DataFrame if no records

# Show DataFrame columns for the first available record set
for rs_id, df in dataframes.items():
    if not df.empty:
        first_record_set_id = rs_id
        print(f"First populated record set: {first_record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Example: Filter, normalize, and group using field @id references
import numpy as np
rs_id = first_record_set_id
df = dataframes[rs_id]

# List numeric columns as candidates
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns in record set {rs_id}: {numeric_cols}")
if numeric_cols:
    numeric_field = numeric_cols[0]  # Pick first numeric field by @id
    threshold = df[numeric_field].quantile(0.5) if not df[numeric_field].isnull().all() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std if std != 0 else filtered_df[numeric_field]
    print(f"Normalized {numeric_field} (z-score):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field (choose one with few unique values)
    group_candidates = df.columns.difference([numeric_field])
    group_field = None
    for col in group_candidates:
        if df[col].nunique() < 10:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram for the numeric field in the selected record set
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field} in record set {rs_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if exists
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field} (record set {rs_id})')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset schema was loaded and key record sets and fields (`@id`) enumerated.
- Sample data from one record set was visualized; numeric distributions were reviewed.
- Typical EDA steps (filtering, normalization, grouping) were demonstrated. For specialized analyses, consult the Croissant schema and documentation for detailed `@id` mapping to domain meanings.